# Reinforcement learning on a sugar module

This tutorial trains a policy with REINFORCE directly against a reactive module authored
in `zrth.sugar` — no gymnasium environment, no extraction. The module is the single
source of truth for the dynamics: the same object steps the rollouts during training
and, at the end, is verified with Z3 together with the learned policy.

## The environment: a corridor

The agent lives on a line of positions $0 \dots 7$ and starts at $0$. Its action is an
external real input `a`: positive means *step right*, otherwise *step left*, and the
position is clamped to the corridor. The goal is the right end, $x = 7$.

The whole environment is one sugar module: `x` is the controlled variable, `a` the
external input, and the clamp is written with `relu`.

In [ ]:
import torch
import torch.nn as nn

from zrth import LRA, Real, Var, X
from zrth import sugar
from zrth.sugar import ite, relu

N = 7.0


class Corridor(sugar.Module):
    def init(self, a):
        return 0.0

    def update(self, x, a):
        step = ite(a > 0.0, x + 1.0, x - 1.0)
        return N - relu(N - relu(step))  # clamp to [0, N]


x, a = Var(Real([1, 1])), Var(Real([1, 1]))
env = Corridor(ctrl=(x,), extl=(a,), theory=LRA)
print(env.with_varnames({x: "x", a: "a"}))

module
  external
    a : Real([1,1])
  interface
    x : Real([1,1])
  atom controls x reads x, a
  init
    #6 := [[0]] 
    X(x) := Id #6
  delay
    d(x) := ZERO 
  update
    #7 := [[0]] 
    #8 := Gt (a, #7)
    #9 := [[1]] 
    #10 := Add (x, #9)
    #11 := [[1]] 
    #12 := Sub (x, #11)
    #13 := Ite (#8, #10, #12)
    #14 := ReLU #13
    #15 := [[7]] 
    #16 := Sub (#15, #14)
    #17 := ReLU #16
    #18 := [[7]] 
    #19 := Sub (#18, #17)
    X(x) := Id #19



## Stepping the module

`zrth.eval` executes the module's blocks over torch tensors: `reset` runs the init
block, `step` seeds the latched wires of `x` and `a` and runs the update block. A `Var`
and its latched wire are interchangeable dictionary keys, so the state dict is keyed by
the variables directly.

In [ ]:
from zrth.eval import execute_init, execute_update


def reset():
    state = {}
    execute_init(state, env.atoms)
    return state[X(x)]


def step(x_val, a_val):
    state = {x: x_val, a: a_val}
    execute_update(state, env.atoms)
    return state[X(x)]


s = reset()
print("start:", s.item(), "| one step right:", step(s, torch.tensor([[1.0]])).item())

start: 0.0 | one step right: 1.0


## The policy

A tiny network scores the current position; a positive score means *go right*. Wrapping
it in `zrth.torch.Module` gives the network a symbolic twin whose `Linear` terms hold
references to the **live** weight tensors — after training, the same object is ready for
verification with no export step. During training the action is sampled from
$\mathrm{Bernoulli}(\sigma(\mathrm{score}))$.

In [ ]:
from zrth.torch import Module as TorchModule


class PolicyNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 4)
        self.fc2 = nn.Linear(4, 1)

    def forward(self, s):
        return self.fc2(torch.relu(self.fc1(s)))


torch.manual_seed(0)
policy = TorchModule(PolicyNN(), combinatorial=True)
print(policy.with_varnames({}))

module
  external
    #22 : Real([1,1])
  interface
    #26 : Real([1,1])
  atom controls #26 awaits #22
  init
    #29 := Transpose X(#22)
    #30 := Linear([[-0.007486820220947266], [0.5364435911178589], [-0.8230451345443726], [-0.7359390258789063]], [[-0.3851543664932251], [0.26815736293792725], [-0.019813179969787598], [0.7928894758224487]]) #29
    #31 := Transpose #30
    #32 := ReLU #31
    #33 := Transpose #32
    #34 := Linear([[-0.04437202215194702, 0.1323062777519226, -0.15110653638839722, -0.09828269481658936]], [[-0.4776742458343506]]) #33
    #35 := Transpose #34
    X(#26) := Id #35
  delay
    d(#26) := ZERO 
  update
    #29 := Transpose X(#22)
    #30 := Linear([[-0.007486820220947266], [0.5364435911178589], [-0.8230451345443726], [-0.7359390258789063]], [[-0.3851543664932251], [0.26815736293792725], [-0.019813179969787598], [0.7928894758224487]]) #29
    #31 := Transpose #30
    #32 := ReLU #31
    #33 := Transpose #32
    #34 := Linear([[-0.04437202215194702, 0.1323

## Training: REINFORCE

Plain REINFORCE, in the notebook, on rollouts stepped through the module: the per-step
reward is the progress $x_{t+1} - x_t$ plus a bonus for reaching the goal, returns-to-go
are normalized as a baseline, and the loss is the usual
$-\sum_t \log \pi(a_t \mid x_t) \, G_t$.

In [ ]:
opt = torch.optim.Adam(policy.parameters(), lr=0.05)
GAMMA, EPISODES, MAX_STEPS = 0.99, 300, 20

for ep in range(EPISODES):
    s, logps, rewards = reset(), [], []
    for _ in range(MAX_STEPS):
        dist = torch.distributions.Bernoulli(torch.sigmoid(policy(s).squeeze()))
        act = dist.sample()
        logps.append(dist.log_prob(act))
        s_next = step(s, torch.tensor([[1.0 if act > 0.5 else -1.0]]))
        rewards.append((s_next - s).item() + (2.0 if s_next.item() == N else 0.0))
        s = s_next
        if s.item() == N:
            break

    returns, G = [], 0.0
    for r in reversed(rewards):
        G = r + GAMMA * G
        returns.append(G)
    returns = torch.tensor(list(reversed(returns)))
    returns = (returns - returns.mean()) / (returns.std() + 1e-6)
    loss = -(torch.stack(logps) * returns).sum()
    opt.zero_grad()
    loss.backward()
    opt.step()

    if ep % 50 == 0 or ep == EPISODES - 1:
        print(f"episode {ep:3d}: reached {s.item():.0f} in {len(rewards):2d} steps")

episode   0: reached 4 in 20 steps
episode  50: reached 7 in  7 steps
episode 100: reached 7 in  7 steps


episode 150: reached 7 in  7 steps
episode 200: reached 7 in  7 steps
episode 250: reached 7 in  7 steps


episode 299: reached 7 in  7 steps


## The learned policy in action

A greedy rollout (no sampling: right iff the score is positive):

In [ ]:
s = reset()
path = [s.item()]
while s.item() != N and len(path) <= 10:
    right = policy(s).item() > 0.0
    s = step(s, torch.tensor([[1.0 if right else -1.0]]))
    path.append(s.item())
print(" -> ".join(f"{v:.0f}" for v in path))

0 -> 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 7


## Verifying the environment and the learned policy

Both objects are symbolic modules, so both can be handed to Z3 by executing their update
terms over arrays of Z3 expressions (shaped `[n, m]`, like the sorts they stand for).

Two properties. First, **safety** of the environment itself: for *any* action, the
position stays in the corridor. Second, **progress** of the learned policy: from every
reachable non-target position ($x \in \{0, \dots, 6\}$), the greedy policy moves
right — which means every rollout reaches the target in at most 7 steps.

In [ ]:
import numpy as np
import z3
from zrth.z3 import eval as zeval


def run_symbolic(module, seed):
    state = dict(seed)
    for atom in module.atoms:
        for term in atom.update:
            state.update(zip(term.write, zeval(term.itype, [state[w] for w in term.read])))
    return state


x_sym, a_sym = z3.Real("x"), z3.Real("a")

state = run_symbolic(env, {x: np.array([[x_sym]]), a: np.array([[a_sym]])})
x_next = state[X(x)].item()

solver = z3.Solver()
solver.add(x_sym >= 0, x_sym <= N)
solver.add(z3.Or(x_next < 0, x_next > N))  # negate safety
if solver.check() == z3.unsat:
    print(f"VERIFIED: x stays in [0, {N:.0f}] for every action")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: x stays in [0, 7] for every action


In [ ]:
r = run_symbolic(policy, {X(list(policy.extl)[0]): np.array([[x_sym]])})
score = r[X(list(policy.intf)[0])].item()

non_target = z3.Or(*[x_sym == float(i) for i in range(int(N))])
solver = z3.Solver()
solver.add(non_target)
solver.add(score <= 0)  # negate: the policy does not move right
if solver.check() == z3.unsat:
    print("VERIFIED: the policy moves right from every non-target position")
    print(f"=> from any start, the target is reached in at most {int(N)} steps")
else:
    print("COUNTEREXAMPLE:", solver.model())

VERIFIED: the policy moves right from every non-target position
=> from any start, the target is reached in at most 7 steps


## Where this goes

One object played both roles: the module stepped the training rollouts and answered the
verifier. The same recipe extends to richer discrete systems today; continuous and
hybrid environments (a pendulum's flow) additionally need a simulator for the `delay`
blocks — the integration story tracked as future work.